# Binary Skin Cancer Classification (Benign vs Malignant) — ConvNeXt / EfficientNetV2 (timm)

This notebook trains a **non‑ResNet** model for **binary** skin lesion classification (benign vs malignant) using **transfer learning**.

Why these models?
- **ConvNeXt** modernizes classic ConvNets and can be very strong on image classification.  
- **EfficientNetV2** targets strong accuracy with efficient training.

We use **timm** (PyTorch Image Models) to load pretrained backbones and fine-tune them.

## Expected dataset folder structure (fanconic Kaggle dataset)
Set `DATA_DIR` to the folder that contains:

```
data/
  train/
    benign/
    malignant/
  test/
    benign/
    malignant/
```

If your dataset path differs, adjust `TRAIN_DIR` and `TEST_DIR` below.


## 0) Install dependencies (run once)

In [ ]:

!pip -q install timm scikit-learn pillow tqdm


## 1) Imports, device, reproducibility

In [1]:

import os, random
from dataclasses import dataclass
from typing import Dict, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler

import timm
from torchvision import transforms
from torchvision.datasets import ImageFolder

from tqdm import tqdm

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score
)

import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)

import pandas as pd

from PIL import Image

from torch.utils.data import Dataset


Device: cuda
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU


## 2) Config

In [ ]:
@dataclass
class CFG:
    # ---- DATA ----
    OUT_ROOT: str = "combined_isic_binary"

    # ---- MODEL ----
    # Non-ResNet options (timm):
    #   - "convnext_tiny.in12k_ft_in1k"        (recommended on 4GB VRAM)
    #   - "tf_efficientnetv2_s.in21k_ft_in1k"  (also strong)
    MODEL_NAME: str = "convnext_tiny.in12k_ft_in1k"
    IMG_SIZE: int = 224

    # ---- TRAINING ----
    EPOCHS: int = 20
    BATCH_SIZE: int = 16        # for 4GB VRAM you may need 8 or 16
    LR: float = 3e-4
    WEIGHT_DECAY: float = 0.02
    NUM_WORKERS: int = 0        # keep 0 on Windows + Jupyter stability

    # ---- IMBALANCE ----
    USE_WEIGHTED_SAMPLER: bool = True
    LABEL_SMOOTHING: float = 0.05

    # ---- AMP ----
    USE_AMP: bool = True

cfg = CFG()
print(cfg)

OUT_ROOT = Path(cfg.OUT_ROOT)
train_csv = OUT_ROOT / "train.csv"
val_csv = OUT_ROOT / "val.csv"
test_csv = OUT_ROOT / "test.csv"
print("OUT_ROOT:", OUT_ROOT)


## 3) Transforms

In [3]:

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(cfg.IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tfms = transforms.Compose([
    transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


## 4) Load datasets + stratified train/val split

In [ ]:
# Manifest-based datasets
train_df = pd.read_csv(train_csv)
val_df = pd.read_csv(val_csv)
test_df = pd.read_csv(test_csv)

class_names = ["benign", "malignant"]
class_to_idx = {c:i for i,c in enumerate(class_names)}

class ManifestDataset(Dataset):
    def __init__(self, df, root: Path, transform=None):
        self.df = df.reset_index(drop=True)
        self.root = Path(root)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.root / row["filepath"]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        label = int(row["label"])
        return img, label

train_ds = ManifestDataset(train_df, OUT_ROOT, transform=train_tfms)
val_ds   = ManifestDataset(val_df, OUT_ROOT, transform=eval_tfms)
test_ds  = ManifestDataset(test_df, OUT_ROOT, transform=eval_tfms)

print("Train size:", len(train_ds))
print("Val size  :", len(val_ds))
print("Test size :", len(test_ds))

train_targets = train_df["label"].values
val_targets   = val_df["label"].values
test_targets  = test_df["label"].values

for name, arr in [("train", train_targets), ("val", val_targets), ("test", test_targets)]:
    counts = {class_names[i]: int((arr==i).sum()) for i in range(len(class_names))}
    print(name, counts)


## 5) DataLoaders (with optional WeightedRandomSampler)

In [ ]:

def make_loader(dataset, batch_size, shuffle=False, sampler=None):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle if sampler is None else False,
        sampler=sampler,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=(device.type=="cuda"),
        drop_last=False
    )

sampler = None
if cfg.USE_WEIGHTED_SAMPLER:
    counts = np.bincount(train_targets, minlength=len(class_names)).astype(np.float32)
    class_w = 1.0 / np.maximum(counts, 1.0)
    sample_w = class_w[train_targets]
    sampler = WeightedRandomSampler(weights=sample_w, num_samples=len(sample_w), replacement=True)
    print("Using WeightedRandomSampler. class_counts:", counts)

train_loader = make_loader(train_ds, cfg.BATCH_SIZE, shuffle=True, sampler=sampler)
val_loader   = make_loader(val_ds, cfg.BATCH_SIZE, shuffle=False)
test_loader  = make_loader(test_ds, cfg.BATCH_SIZE, shuffle=False)


## 6) Build model (ConvNeXt / EfficientNetV2 via timm)

In [ ]:

num_classes = len(class_names)
assert num_classes == 2, f"Expected 2 classes, found {num_classes}. Check folder names."

model = timm.create_model(cfg.MODEL_NAME, pretrained=True, num_classes=num_classes).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=cfg.LABEL_SMOOTHING)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)

steps_per_epoch = len(train_loader)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=cfg.LR,
    epochs=cfg.EPOCHS,
    steps_per_epoch=steps_per_epoch
)

scaler = torch.cuda.amp.GradScaler(enabled=(cfg.USE_AMP and device.type=="cuda"))

print("Model:", cfg.MODEL_NAME)
print("Params (M):", sum(p.numel() for p in model.parameters())/1e6)


## 7) Training + evaluation utilities

In [ ]:

@torch.no_grad()
def predict_probs(loader):
    model.eval()
    all_probs, all_y = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(cfg.USE_AMP and device.type=='cuda')):
            logits = model(x)
            probs = torch.softmax(logits, dim=1)
        all_probs.append(probs.detach().cpu().numpy())
        all_y.append(y.detach().cpu().numpy())
    return np.concatenate(all_probs, axis=0), np.concatenate(all_y, axis=0)

@torch.no_grad()
def evaluate(loader, threshold=0.5):
    probs, y_true = predict_probs(loader)
    pos_idx = 1  # positive class assumed to be index 1
    y_prob_pos = probs[:, pos_idx]
    y_pred_thr = (y_prob_pos >= threshold).astype(int)
    y_pred_arg = probs.argmax(axis=1)

    acc = accuracy_score(y_true, y_pred_arg)
    bacc = balanced_accuracy_score(y_true, y_pred_arg)
    f1 = f1_score(y_true, y_pred_thr)
    prec = precision_score(y_true, y_pred_thr, zero_division=0)
    rec = recall_score(y_true, y_pred_thr, zero_division=0)

    try:
        roc_auc = roc_auc_score(y_true, y_prob_pos)
    except Exception:
        roc_auc = float('nan')
    try:
        ap = average_precision_score(y_true, y_prob_pos)
    except Exception:
        ap = float('nan')

    return {
        "top1_acc": acc,
        "balanced_acc": bacc,
        "f1@0.5": f1,
        "precision@0.5": prec,
        "recall@0.5": rec,
        "roc_auc": roc_auc,
        "pr_auc": ap
    }

def train_one_epoch(epoch: int):
    model.train()
    running = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.EPOCHS}", leave=False)
    for x, y in pbar:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(cfg.USE_AMP and device.type=='cuda')):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running += loss.item() * x.size(0)
        pbar.set_postfix({"loss": float(loss.item()), "lr": float(scheduler.get_last_lr()[0])})

    return running / len(train_loader.dataset)


## 8) Train loop (save best by validation F1@0.5)

In [ ]:

history = {"train_loss": [], "val_top1": [], "val_f1": [], "val_auc": []}
best_f1 = -1.0
best_path = f"best_{cfg.MODEL_NAME.replace('.', '_')}.pt"

for epoch in range(cfg.EPOCHS):
    train_loss = train_one_epoch(epoch)
    val_metrics = evaluate(val_loader, threshold=0.5)

    history["train_loss"].append(train_loss)
    history["val_top1"].append(val_metrics["top1_acc"])
    history["val_f1"].append(val_metrics["f1@0.5"])
    history["val_auc"].append(val_metrics["roc_auc"])

    print(f"[Epoch {epoch+1:02d}] train_loss={train_loss:.4f} | "
          f"val_top1={val_metrics['top1_acc']:.4f} val_f1@0.5={val_metrics['f1@0.5']:.4f} "
          f"val_auc={val_metrics['roc_auc']:.4f}")

    if val_metrics["f1@0.5"] > best_f1:
        best_f1 = val_metrics["f1@0.5"]
        torch.save(model.state_dict(), best_path)
        print("  ✓ saved best:", best_path)

print("Best val F1@0.5:", best_f1, "checkpoint:", best_path)


## 9) Plot learning curves

In [ ]:

plt.figure()
plt.plot(history["train_loss"], label="train_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.legend()
plt.show()

plt.figure()
plt.plot(history["val_top1"], label="val_top1_acc")
plt.plot(history["val_f1"], label="val_f1@0.5")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Validation Metrics")
plt.legend()
plt.show()

plt.figure()
plt.plot(history["val_auc"], label="val_roc_auc")
plt.xlabel("Epoch")
plt.ylabel("AUC")
plt.title("Validation ROC-AUC")
plt.legend()
plt.show()


## 10) Final evaluation on test set + reports + plots

In [ ]:

model.load_state_dict(torch.load(best_path, map_location=device))

test_metrics = evaluate(test_loader, threshold=0.5)
print("TEST metrics:", test_metrics)

probs, y_true = predict_probs(test_loader)
y_pred_argmax = probs.argmax(axis=1)

print("\nClassification report (argmax):")
print(classification_report(y_true, y_pred_argmax, target_names=class_names, zero_division=0))

cm = confusion_matrix(y_true, y_pred_argmax)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(values_format="d")
plt.title("Confusion Matrix (Test, argmax)")
plt.show()

pos_idx = 1
y_prob_pos = probs[:, pos_idx]

fpr, tpr, _ = roc_curve(y_true, y_prob_pos)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
plt.plot([0,1],[0,1],"--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (Test)")
plt.legend()
plt.show()

prec, rec, _ = precision_recall_curve(y_true, y_prob_pos)
ap = average_precision_score(y_true, y_prob_pos)

plt.figure()
plt.plot(rec, prec, label=f"AP = {ap:.4f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve (Test)")
plt.legend()
plt.show()


## 11) Try another model quickly (one-liner)
Set `cfg.MODEL_NAME` to e.g. `tf_efficientnetv2_s.in21k_ft_in1k` and rerun from **Section 6** onward.

In [ ]:

# Example:
# cfg.MODEL_NAME = "tf_efficientnetv2_s.in21k_ft_in1k"
# (then rerun from Section 6)
